# Parte 2: Ahora sí, los métodos automáticos

En la Parte 1 elegiste features **a mano**, por intuición. Ahora vas a usar tres métodos de selección de features reales — los mismos que viste en la presentación de "Evaluación y diagnóstico" (Filters y Wrappers) — y al final vas a comparar si le ganaron, o no, a tu propio criterio.

Los tres métodos que vas a aplicar:

- **Sequential Forward Selection** (wrapper): empieza sin features y va agregando, una por una, la que más mejora el modelo en cada paso.
- **RFECV — Recursive Feature Elimination con validación cruzada** (wrapper): empieza con todas las features y va quitando, una por una, la que menos aporta — pero a diferencia de un RFE normal, usa cross-validation para decidir por sí solo cuántas features quedarse, no se lo dices tú.
- **SelectKBest con correlación** (filter): elige directamente las `k` features más correlacionadas con la variable objetivo, sin entrenar ningún modelo de por medio.

## Cómo vas a trabajar

Para cada método, en vez de darte el código completo, tienes un `# TODO` con el link a la documentación oficial de scikit-learn. Tu trabajo es completar la línea que falta. **Antes de correr tu código, escribe tu predicción** en la celda de markdown correspondiente — qué features crees que va a elegir el método, y por qué. Eso es lo que hace que esto sea un ejercicio, no una receta.

In [1]:
# ============================================================
# PREPARACIÓN — NO MODIFICAR
# (recarga el mismo dataset y el mismo split fijo de la Parte 1,
#  para que los resultados sigan siendo comparables)
# ============================================================
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

diabetes = load_diabetes(as_frame=True)
df = diabetes.frame
X = df.drop(columns=['target'])
y = df['target']

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print("Datos recargados. Tamaño de entrenamiento:", X_train.shape)

Datos recargados. Tamaño de entrenamiento: (265, 10)


In [2]:
# ============================================================
# PREPARACIÓN 3 — NO MODIFICAR
# ============================================================
historial_metodos = []

def evaluar_metodo(nombre, selector):
    """Ajusta el selector que le pases sobre el conjunto de entrenamiento,
    entrena una regresión lineal con las features que eligió, y evalúa
    contra VALIDACIÓN. El selector debe ser un objeto de sklearn.feature_selection
    ya construido (SequentialFeatureSelector, RFE o SelectKBest), sin ajustar (.fit) todavía."""
    selector.fit(X_train, y_train)
    features_elegidas = list(X_train.columns[selector.get_support()])
    modelo = LinearRegression()
    modelo.fit(X_train[features_elegidas], y_train)
    y_pred = modelo.predict(X_val[features_elegidas])
    mse = mean_squared_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)
    historial_metodos.append({'metodo': nombre, 'features': features_elegidas, 'mse': mse, 'r2': r2})
    print(f"[{nombre}]")
    print(f"  Features elegidas ({len(features_elegidas)}): {features_elegidas}")
    print(f"  MSE (validation): {mse:.2f}")
    print(f"  R^2 (validation): {r2:.4f}")
    return selector, mse, r2

_cerrado_metodo = {'listo': False}

def cerrar_seleccion_metodo(nombre, selector):
    """Evalúa UNA SOLA VEZ contra el conjunto de PRUEBA, con el selector
    que consideres tu mejor método. Esta es tu selección automática oficial."""
    if _cerrado_metodo['listo']:
        print("Ups — ya cerraste tu método ganador antes. No se puede repetir.")
        return
    selector.fit(X_train, y_train)
    features_elegidas = list(X_train.columns[selector.get_support()])
    modelo = LinearRegression()
    modelo.fit(X_train[features_elegidas], y_train)
    y_pred_test = modelo.predict(X_test[features_elegidas])
    mse_test = mean_squared_error(y_test, y_pred_test)
    r2_test = r2_score(y_test, y_pred_test)
    _cerrado_metodo['listo'] = True
    print("="*50)
    print(f"RESULTADO FINAL AUTOMÁTICO — {nombre} (conjunto de prueba)")
    print("="*50)
    print(f"Features: {features_elegidas}")
    print(f"MSE test: {mse_test:.2f}")
    print(f"R^2 test: {r2_test:.4f}")
    return mse_test, r2_test

print("Listo. Ahora sigue con los tres métodos de abajo.")

Listo. Ahora sigue con los tres métodos de abajo.


## A. Sequential Forward Selection (wrapper)

Documentación: https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SequentialFeatureSelector.html

**Tu predicción (escríbela aquí antes de correr el código):** ¿cuántas y cuáles features crees que va a elegir? ¿Por qué?

*(escribe tu respuesta en esta celda)*

In [3]:
# TODO: completa la construcción del selector.
# Necesitas:
#   - un estimator (usa LinearRegression())
#   - decidir cuántas features quieres que elija (n_features_to_select) — prueba con un número, tú decides
#   - direction='forward'
#
# from sklearn.feature_selection import SequentialFeatureSelector
#
# estimator = ...
# selector_forward = SequentialFeatureSelector(...)
#
# evaluar_metodo("Sequential Forward Selection", selector_forward)


## B. RFECV — Recursive Feature Elimination con validación cruzada (wrapper)

Documentación: https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.RFECV.html

Esta versión es distinta a los otros dos métodos: **no le dices cuántas features quieres**. RFECV prueba distintos tamaños internamente usando cross-validation, y decide por sí solo cuál número de features es el mejor.

**Tu predicción:** además de qué features crees que va a elegir, predice **cuántas** features va a decidir usar por su cuenta (un número entre 1 y 9). ¿Por qué ese número?

*(escribe tu respuesta en esta celda)*

In [4]:
# TODO: completa la construcción del selector.
# Necesitas:
#   - un estimator (usa LinearRegression())
#   - cv (número de folds para la validación cruzada interna, por ejemplo 5)
#
# A diferencia de los otros métodos, aquí NO le das n_features_to_select —
# RFECV decide solo cuántas features usar.
#
# from sklearn.feature_selection import RFECV
#
# estimator = ...
# selector_rfecv = RFECV(...)
#
# evaluar_metodo("RFECV", selector_rfecv)


## C. SelectKBest con correlación (filter)

Documentación: https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectKBest.html
(usa como función de score `r_regression`, de `sklearn.feature_selection`)

**Tu predicción:** este método NO entrena ningún modelo para elegir features, solo mide correlación directa con la variable objetivo. ¿Crees que le va a ir mejor, peor o parecido a los dos métodos anteriores? ¿Por qué?

*(escribe tu respuesta en esta celda)*

In [5]:
# TODO: completa la construcción del selector.
# Necesitas:
#   - la función de score r_regression
#   - k (cuántas features quieres que elija)
#
# from sklearn.feature_selection import SelectKBest, r_regression
#
# selector_filtro = SelectKBest(...)
#
# evaluar_metodo("SelectKBest (correlación)", selector_filtro)


## Cierra tu método ganador — ¡también solo se puede hacer una vez!

De los tres métodos que probaste, ¿cuál dio el mejor MSE en validación? Cierra ese con `cerrar_seleccion_metodo(...)`, pasando el mismo nombre y selector que usaste arriba:

```python
cerrar_seleccion_metodo("RFECV", selector_rfecv)
```

In [6]:
# Tu método ganador:
# cerrar_seleccion_metodo(..., ...)

## Para discutir

1. ¿Tu selección manual de la Parte 1 le ganó al mejor método automático de la Parte 2, o fue al revés? ¿Por cuánto?
2. ¿Los tres métodos automáticos eligieron las mismas features entre sí? Si no, ¿por qué crees que un wrapper (que sí entrena modelos) puede elegir distinto que un filtro (que solo mide correlación)?
3. ¿Alguno de los métodos automáticos incluyó `sex`, la variable con correlación casi nula que vimos en la Parte 1? Si sí, ¿por qué un método "inteligente" caería en eso?
4. ¿Tu predicción de qué features iba a elegir cada método se cumplió? Si no, ¿qué fue lo que no esperabas?